<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/main/Exercice_XP_W8D3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installation du paquet MCP
!pip install "mcp[cli]"

# Vérification
!mcp --help

                                                                                
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 MCP development tools                                                          
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --help          Show this message and exit.                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ version   Show the MCP version.                                              │
│ dev       Run an MCP server with the MCP Inspector.                          │
│ run       Run an MCP server.                                                 │
│ install   Install an MCP s

In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Demo")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    return a + b

@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting server.py


In [ ]:
%%writefile client.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command="mcp",
    args=["run", "server.py"],
    env=None
)

def extract_content(payload):
    """Best-effort to pull text from MCP responses."""
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    if hasattr(payload, "content"):
        return payload.content
    return str(payload)

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Lister les ressources
            resources = await session.list_resources()
            print("=== Ressources disponibles ===")
            for res in resources.resources:
                print(f"  - URI: {res.uri} (nom: {res.name})")

            # Lister les outils
            tools = await session.list_tools()
            print("\n=== Outils disponibles ===")
            for tool in tools.tools:
                print(f"  - Nom: {tool.name} (description: {tool.description})")

            # Lire la ressource greeting://hello
            print("\n=== Lecture de greeting://hello ===")
            result_resource = await session.read_resource("greeting://hello")
            texte = extract_content(result_resource)
            print(f"  Contenu: {texte}")

            # Appeler l'outil add avec a=1, b=7
            print("\n=== Appel de l'outil add(1, 7) ===")
            result_tool = await session.call_tool("add", arguments={"a": 1, "b": 7})
            resultat = extract_content(result_tool)
            print(f"  Résultat: {resultat}")

if __name__ == "__main__":
    asyncio.run(run())

Overwriting client.py


In [ ]:
!python client.py

[07/08/26 15:33:34] INFO     Processing request of type            ]8;id=232721;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=288311;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListResourcesRequest                               
=== Ressources disponibles ===
                    INFO     Processing request of type            ]8;id=483851;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=634351;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   

=== Outils disponibles ===
- add (description: Additionne deux entiers.)

=== Lecture de la ressource greeting://hello ===
                    INFO     Processing request of type            ]8;id=809530;file:///usr/local/lib/python3.12/dis

In [ ]:
import subprocess
import time

# Lancer le serveur en arrière-plan
server_process = subprocess.Popen(["mcp", "run", "server.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(2)  # laisser le temps au serveur de démarrer

# Puis lancer le client (code ci-dessus) en parallèle ?